# Next step: step 3, seeds
**What it adds:** a `seed` argument, so a run can be repeated exactly. Before, the disk particles started
from random positions that changed every run, so two runs of the same model could never be compared
particle by particle.

**How it connects:**
- **Below, in JSPAM:** all of SPAM's random numbers come from one function, `randm()` in `setup_module.f90`,
  which calls Fortran's built-in `random_number`. It's used only to place the disk particles. The generator's
  starting point is set by `RANDOM_SEED`, so seeding only needs changes to our own `simr_interface.f90`, not
  to Wallin's modules.
- **Here:** `sim.simulate(..., seed=...)` passes the seed through to `jspam.run_collision`.
- **Above:** `simr` will give each model its own seed, worked out from one run seed (step 10), so a whole run
  repeats exactly, however many workers it uses.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed
**JSPAM, `fortran/simr_interface.f90`:** three new subroutines. `run_collision` no longer calls
`RANDOM_SEED()` itself, so Python decides how it's seeded before each run.
```fortran
! How many integers the compiler's random number generator needs for a seed.
subroutine seed_size(nseed)
  integer, intent(out) :: nseed
  call RANDOM_SEED(size=nseed)
end subroutine seed_size

! Set the generator's seed, so the disk particles start the same way every run.
subroutine set_seed(values, nseed)
  integer, intent(in) :: nseed
  integer, dimension(nseed), intent(in) :: values
  call RANDOM_SEED(put=values)
end subroutine set_seed

! Seed the generator from the operating system, so every run differs (the original behavior).
subroutine seed_from_system()
  call RANDOM_SEED()
end subroutine seed_from_system
```
**JSPAM, `src/jspam/__init__.py`:** `run_collision` takes `seed`. gfortran's generator needs several
integers, not one, so `_seed_values` expands one integer into that many with numpy's `SeedSequence`. Even
nearby seeds like 1 and 2 give unrelated particles.
```python
def run_collision(params, n1=1000, n2=1000, potential=EXTENDED, seed=None):
    ...
    if seed is None:
        simr_interface.seed_from_system()
    else:
        simr_interface.set_seed(_seed_values(seed))
    return simr_interface.run_collision(params, potential, n1, n2)


def _seed_values(seed):
    size = simr_interface.seed_size()
    state = np.random.SeedSequence(seed).generate_state(size, dtype=np.uint32)
    return state.view(np.int32)
```
**`sim.py`:** `simulate` takes `seed` and passes it on.
```python
def simulate(params, n1=1000, n2=1000, seed=None):
    return jspam.run_collision(params, n1=n1, n2=n2, seed=seed)
```

In [ ]:
%matplotlib inline
import jspam
import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
params = sim.parse_model_string(MODEL_STRING)
N1, N2 = 1000, 1000

## 1. The seed SPAM needs
gfortran's generator takes a seed of several integers. `_seed_values` builds them from one number.

In [ ]:
print("integers in a gfortran seed:", jspam.simr_interface.seed_size())
print("seed 1 ->", jspam._seed_values(1))
print("seed 2 ->", jspam._seed_values(2))

## 2. Same seed, same particles
Exactly equal, not just close: `np.array_equal` checks every bit. That holds on the same build and machine;
another compiler or CPU may differ in the last digits.

In [ ]:
a = sim.simulate(params, n1=N1, n2=N2, seed=1)
b = sim.simulate(params, n1=N1, n2=N2, seed=1)
print("identical:", np.array_equal(a, b))

## 3. Different seed, different particles, same orbit

In [ ]:
c = sim.simulate(params, n1=N1, n2=N2, seed=2)
print("disk particles match:      ", np.allclose(a[:-1], c[:-1]))
print("secondary's center matches:", np.allclose(a[-1], c[-1]))

## 4. No seed: different every run (Wallin's original behavior)

In [ ]:
d = sim.simulate(params, n1=N1, n2=N2)
e = sim.simulate(params, n1=N1, n2=N2)
print("two unseeded runs match:", np.allclose(d[:-1], e[:-1]))

## 5. What a seed changes, by eye
Seeds 1 and 2 side by side, plus seed 1 again. The tidal features are the same; only which particles trace
them differs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for ax, (title, run) in zip(axes, [("seed 1", a), ("seed 2", c), ("seed 1 again", b)]):
    ax.scatter(run[:N1, 0], run[:N1, 1], s=1)
    ax.scatter(run[N1:N1 + N2, 0], run[N1:N1 + N2, 1], s=1)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.show()